In [0]:
from pyspark.sql.functions import current_date, current_timestamp, lit, regexp_extract, to_date

# Source: NHFR active facilities CSV in the PSA landing zone
nhfr_source_path = (
    "/Volumes/workspace/default/ftw-b12-de/groups/g5_landing/facilities/"
    "nhfr_activefacilities_asof_20260929.csv"
)

# Preserve the source rows and values without inferring a new schema.
nhfr_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", False)
         .csv(nhfr_source_path)
    .withColumn("source_file", lit(nhfr_source_path.rsplit("/", 1)[-1]))
    .withColumn("as_of_date", to_date(regexp_extract(lit(nhfr_source_path.rsplit("/", 1)[-1]), r"(?i)asof_?(\d{8})", 1), "yyyyMMdd"))
    .withColumn("extract_date", current_date())
    .withColumn("ingested_at", current_timestamp())
)

# Write to bronze Delta table (full refresh)
# Column mapping is enabled because the CSV column names contain
# spaces and special characters (e.g., "Street Name and #")
(
    nhfr_df.write
           .format("delta")
           .mode("overwrite")
           .option("delta.columnMapping.mode", "name")
           .option("delta.minReaderVersion", "2")
           .option("delta.minWriterVersion", "5")
           .saveAsTable("public_health.bronze.active_facilities")
)